# Day 77: Pandas Advanced -- Index Types, Time Series, Categorical & Multi-Index

This notebook covers advanced pandas index types with a focus on:

- **DatetimeIndex** -- time series creation, resampling, timezone handling
- **CategoricalIndex** -- categorical data, ordered categories, group-by optimization
- **MultiIndex** -- hierarchical indexing, multi-level groupby, pivot operations
- **C++ comparison** -- how pandas achieves in one line what C++ requires dozens of
- **Enterprise examples** -- real-world financial, HR, and supply-chain scenarios

---

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd

print(f'pandas version: {pd.__version__}')
print(f'numpy  version: {np.__version__}')

---
## 1. DatetimeIndex & Time Series

`DatetimeIndex` is the most important index type in pandas for time series analysis.  
Key functions: `pd.date_range()`, `asfreq()`, `resample()`, `tz_localize()`, `tz_convert()`.

### 1.1 Creating a DatetimeIndex with `date_range()`

In [ ]:
# -- Specify start, end, and number of periods (evenly spaced) --
idx_periods = pd.date_range('2025-01-01', '2025-06-30', periods=10)
print('10 evenly-spaced dates (Jan-Jun 2025):')
print(idx_periods)
print()

In [ ]:
# -- Specify a frequency string --
idx_weekly = pd.date_range('2025-01-01', '2025-06-30', freq='W-MON')   # every Monday
idx_monthly = pd.date_range('2025-01-01', '2025-12-31', freq='MS')     # month-start
idx_12h = pd.date_range('2025-01-01', periods=8, freq='12H')           # every 12 hours

print('Weekly (Monday):', idx_weekly[:5].tolist(), '...')
print('Monthly start :', idx_monthly[:5].tolist(), '...')
print('12-hourly     :', idx_12h.tolist())

### 1.2 DateOffset Arithmetic

`DatetimeIndex` supports arithmetic with `pd.DateOffset` for time shifting.

In [ ]:
idx = pd.date_range('2025-01-05', periods=5, freq='W-SUN')
print('Original (Sundays):')
print(idx)
print()

print('Shift back 2 days (Fridays):')
print(idx - pd.DateOffset(days=2))
print()

print('Shift forward 2h 10min:')
print(idx + pd.DateOffset(hours=2, minutes=10))

### 1.3 Enterprise Example: Simulated Daily Stock Data

We simulate one year of daily OHLCV data for a fictional tech company, then demonstrate `asfreq`, `resample`, and timezone conversion.

In [ ]:
np.random.seed(42)

# Generate business-day dates for 2025
dates = pd.bdate_range('2025-01-01', '2025-12-31')  # ~261 trading days
n = len(dates)

# Simulate a random-walk price series
close = 100.0 + np.cumsum(np.random.randn(n) * 1.5)
high  = close + np.abs(np.random.randn(n))
low   = close - np.abs(np.random.randn(n))
open_ = close + np.random.randn(n) * 0.5
volume = np.random.randint(500_000, 5_000_000, n)

stock_df = pd.DataFrame({
    'Open':   open_.round(2),
    'High':   high.round(2),
    'Low':    low.round(2),
    'Close':  close.round(2),
    'Volume': volume
}, index=dates)
stock_df.index.name = 'Date'

print(f'Stock data shape: {stock_df.shape}')
stock_df.head(10)

### 1.4 `asfreq()` -- Sampling at a Fixed Frequency

Every 5 trading days, grab one sample. Non-trading days become NaN unless we forward-fill.

In [ ]:
sampled = stock_df.asfreq('5B')  # every 5 business days
print('asfreq("5B") -- first 8 rows:')
print(sampled.head(8))
print()

filled = stock_df.asfreq('5B', method='ffill')
print('asfreq("5B", method="ffill") -- first 8 rows:')
print(filled.head(8))

### 1.5 `resample()` -- Time-Based Grouping & Aggregation

`resample` groups data by a time bucket (e.g. monthly, quarterly) and lets you aggregate.

In [ ]:
# Monthly mean of all OHLCV columns
monthly_mean = stock_df.resample('MS').mean()
print('Monthly averages (first 6 months):')
print(monthly_mean.head(6))

In [ ]:
# Multiple aggregations at once
monthly_stats = stock_df['Close'].resample('QS').agg(['mean', 'std', 'min', 'max'])
print('Quarterly Close price statistics:')
print(monthly_stats.round(2))

In [ ]:
# Quarterly total volume + mean close (custom aggregation)
quarterly_summary = stock_df.resample('QS').agg({
    'Close':  'mean',
    'Volume': 'sum'
})
print('Quarterly summary (mean close, total volume):')
print(quarterly_summary.round(2))

### 1.6 Timezone Handling

Use `tz_localize()` to attach a timezone, then `tz_convert()` to shift.

In [ ]:
stock_cn = stock_df.copy()
stock_cn.index = stock_cn.index.tz_localize('Asia/Shanghai')
print('Localized to Asia/Shanghai:')
print(stock_cn.head(3))
print()

stock_us = stock_cn.tz_convert('America/New_York')
print('Converted to America/New_York:')
print(stock_us.head(3))

---
## 2. CategoricalIndex

`CategoricalIndex` is built from **nominal-scale** data. It excels at group-by aggregation and can enforce a custom display order via `reorder_categories()`.

### 2.1 Basic CategoricalIndex

In [ ]:
sales_data = [6, 6, 7, 6, 8, 6]
index = pd.CategoricalIndex(
    data=['Apple', 'Banana', 'Apple', 'Apple', 'Peach', 'Banana'],
    categories=['Apple', 'Banana', 'Peach'],
    ordered=True
)
ser = pd.Series(data=sales_data, index=index)
print('Series with CategoricalIndex:')
print(ser)
print()

In [ ]:
# Group by the categorical index and sum
grouped = ser.groupby(level=0).sum()
print('Grouped sum by category:')
print(grouped)
print()

In [ ]:
# Reorder categories -- changes display order of groupby result
ser.index = index.reorder_categories(['Banana', 'Peach', 'Apple'])
print('After reorder_categories ([Banana, Peach, Apple]):')
print(ser.groupby(level=0).sum())

### 2.2 Enterprise Example: Employee Department & Seniority

A company HR dataset where departments and seniority levels are categorical.  
CategoricalIndex lets us enforce display order (Junior < Mid < Senior) in group-by results.

In [ ]:
np.random.seed(7)
n_emp = 30

departments = ['Engineering', 'Sales', 'Marketing', 'Finance']
seniority = ['Junior', 'Mid', 'Senior']

dept_cat = pd.CategoricalIndex(
    data=np.random.choice(departments, n_emp),
    categories=departments,
    ordered=True
)
seniority_cat = pd.CategoricalIndex(
    data=np.random.choice(seniority, n_emp),
    categories=['Junior', 'Mid', 'Senior'],
    ordered=True
)

hr_df = pd.DataFrame({
    'Department':  dept_cat,
    'Seniority':   seniority_cat,
    'Salary':      (np.random.randint(50, 180, n_emp) * 1000),
    'Performance': np.random.uniform(2.5, 5.0, n_emp).round(2)
})

print('HR Dataset:')
print(hr_df.head(10))
print()

# Average salary by department (ordered by the categorical order)
print('Average salary by department:')
print(hr_df.groupby('Department')['Salary'].mean().round(0).astype(int))
print()

# Average salary by seniority (ordered: Junior < Mid < Senior)
print('Average salary by seniority level:')
print(hr_df.groupby('Seniority')['Salary'].mean().round(0).astype(int))

In [ ]:
# Cross-tabulation: department x seniority, average performance
perf_pivot = hr_df.pivot_table(
    values='Performance',
    index='Department',
    columns='Seniority',
    aggfunc='mean'
)
print('Average Performance by Department x Seniority:')
print(perf_pivot.round(2))

---
## 3. MultiIndex (Hierarchical Indexing)

`MultiIndex` allows multiple levels of row/column labels.  
Creation methods: `from_tuples`, `from_arrays`, `from_product`.

### 3.1 Creating MultiIndex

In [ ]:
# from_tuples
tuples = [(1, 'red'), (1, 'blue'), (2, 'red'), (2, 'blue')]
mi_tuples = pd.MultiIndex.from_tuples(tuples, names=['no', 'color'])
print('from_tuples:')
print(mi_tuples)
print()

# from_arrays
arrays = [[1, 1, 2, 2], ['red', 'blue', 'red', 'blue']]
mi_arrays = pd.MultiIndex.from_arrays(arrays, names=['no', 'color'])
print('from_arrays:')
print(mi_arrays)
print()

# from_product (Cartesian product)
mi_product = pd.MultiIndex.from_product(
    [[1, 2], ['red', 'blue']],
    names=['no', 'color']
)
print('from_product:')
print(mi_product)

In [ ]:
# Series with MultiIndex
np.random.seed(0)
data = np.random.randint(1, 100, 4)
ser_mi = pd.Series(data=data, index=mi_tuples)
print('Series with MultiIndex:')
print(ser_mi)
print()

# Group by level 0 (no)
print('Groupby level=0 (no):')
print(ser_mi.groupby(level=0).sum())
print()

# Group by level 1 (color)
print('Groupby level=1 (color):')
print(ser_mi.groupby(level=1).sum())

### 3.2 MultiIndex in DataFrame -- Student Scores Example

In [ ]:
np.random.seed(42)
stu_ids = np.arange(1001, 1006)
semesters = ['Midterm', 'Final']
index = pd.MultiIndex.from_product((stu_ids, semesters), names=['StudentID', 'Exam'])
courses = ['Chinese', 'Math', 'English']
scores = np.random.randint(60, 101, (10, 3))

student_df = pd.DataFrame(data=scores, columns=courses, index=index)
print('Student scores (MultiIndex on rows):')
print(student_df)

In [ ]:
# Compute weighted average: Midterm 25%, Final 75%
weighted = student_df.groupby(level=0).agg(
    lambda x: x.values[0] * 0.25 + x.values[1] * 0.75
)
print('Weighted average (Midterm 25% + Final 75%):')
print(weighted.round(2))

### 3.3 Enterprise Example: Multi-Region, Multi-Product Sales

A supply-chain scenario: sales data indexed by (Region, Product, Quarter).  
Demonstrates cross-sectional slicing, unstack, and multi-level groupby.

In [ ]:
np.random.seed(99)

regions  = ['North', 'South', 'East', 'West']
products = ['Widget-A', 'Widget-B', 'Widget-C']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

mi = pd.MultiIndex.from_product(
    [regions, products, quarters],
    names=['Region', 'Product', 'Quarter']
)

supply_df = pd.DataFrame({
    'UnitsSold':  np.random.randint(100, 5000, len(mi)),
    'Revenue':    np.random.randint(10_000, 500_000, len(mi)),
    'Cost':       np.random.randint(5_000, 300_000, len(mi))
}, index=mi)
supply_df['Profit'] = supply_df['Revenue'] - supply_df['Cost']

print('Supply-chain dataset (first 12 rows):')
print(supply_df.head(12))

In [ ]:
# Slice: all products in the North region
print('North region data:')
print(supply_df.loc['North'])
print()

In [ ]:
# Slice: Widget-A across all regions, all quarters
print('Widget-A across all regions & quarters:')
print(supply_df.loc[(slice(None), 'Widget-A', slice(None)), :])

In [ ]:
# Total profit by Region and Product (summing over quarters)
profit_by_rp = supply_df.groupby(level=['Region', 'Product'])['Profit'].sum()
print('Total Profit by Region x Product:')
print(profit_by_rp.unstack('Product').round(0).astype(int))

In [ ]:
# Quarterly revenue trend per region
rev_trend = supply_df.groupby(level=['Region', 'Quarter'])['Revenue'].sum().unstack('Quarter')
print('Revenue by Region x Quarter:')
print(rev_trend.round(0).astype(int))

---
## 4. IntervalIndex

Used for binning continuous data into fixed-width intervals (e.g. age groups, price ranges).

In [ ]:
# Basic interval range
idx_iv = pd.interval_range(start=0, end=5)
print('Default (right-closed):')
print(idx_iv)
print()

print('Contains 1.5?', idx_iv.contains(1.5))
print('Overlaps (1.5, 3.5)?', idx_iv.overlaps(pd.Interval(1.5, 3.5)))
print()

# Left-closed
idx_left = pd.interval_range(start=0, end=5, closed='left')
print('Left-closed:')
print(idx_left)
print()

# Date intervals
idx_date = pd.interval_range(
    start=pd.Timestamp('2025-01-01'),
    end=pd.Timestamp('2025-01-04'),
    closed='both'
)
print('Date intervals (both-closed):')
print(idx_date)

In [ ]:
# Enterprise use case: bin customer ages into groups
np.random.seed(12)
ages = np.random.randint(18, 70, 200)

age_bins = pd.cut(ages, bins=[17, 25, 35, 50, 65, 70],
                  labels=['18-25', '26-35', '36-50', '51-65', '66+'])
age_dist = pd.Series(age_bins).value_counts().sort_index()
print('Customer age distribution:')
print(age_dist)

---
## 5. RangeIndex

The default integer index, monotonically increasing.

In [ ]:
np.random.seed(5)
sales_data = np.random.randint(400, 1000, 12)
index = pd.RangeIndex(1, 13, name='Month')
ser_range = pd.Series(data=sales_data, index=index)
print('Monthly sales with RangeIndex:')
print(ser_range)
print()
print('Index type:', type(ser_range.index))

---
## 6. C++ Comparison

Below we show how pandas achieves in a few lines what would require **dozens of lines** in C++ (with STL containers).  
The key insight: pandas provides **vectorized operations on indexed, labeled data** out of the box.

### 6.1 Time Series Resampling

**pandas (3 lines):**
```python
dates = pd.date_range('2025-01-01', periods=365, freq='D')
ts = pd.Series(np.random.randn(365).cumsum(), index=dates)
monthly_avg = ts.resample('MS').mean()
```

**C++ equivalent (conceptual ~40+ lines with STL):**
```cpp
#include <vector>
#include <map>
#include <string>
#include <numeric>
#include <ctime>

// 1. Generate dates manually
std::vector<std::time_t> dates;
std::tm start = {0, 0, 0, 1, 0, 125}; // 2025-01-01
for (int i = 0; i < 365; ++i) {
    std::tm d = start;
    d.tm_mday += i;
    std::mktime(&d);
    dates.push_back(std::mktime(&d));
}

// 2. Generate random walk
std::vector<double> values(365);
double cum = 0.0;
for (int i = 0; i < 365; ++i) {
    cum += /* random normal */ 0.0;  // omitted for brevity
    values[i] = cum;
}

// 3. Group by month and compute mean
std::map<int, std::vector<double>> monthly;
for (int i = 0; i < 365; ++i) {
    std::tm* d = std::localtime(&dates[i]);
    monthly[d->tm_mon].push_back(values[i]);
}

std::map<int, double> monthly_avg;
for (auto& [month, vals] : monthly) {
    double sum = std::accumulate(vals.begin(), vals.end(), 0.0);
    monthly_avg[month] = sum / vals.size();
}
```

pandas handles date generation, grouping, and aggregation in 3 lines with labeled output.  
C++ requires manual date arithmetic, container management, and explicit accumulation logic.

### 6.2 Categorical Grouping

**pandas:**
```python
idx = pd.CategoricalIndex(['A','B','A','A','B'], categories=['A','B'], ordered=True)
s = pd.Series([10, 20, 30, 40, 50], index=idx)
result = s.groupby(level=0).sum()
```

**C++ equivalent (~25 lines):**
```cpp
#include <vector>
#include <string>
#include <map>
#include <algorithm>

std::vector<std::string> cats = {"A", "B", "A", "A", "B"};
std::vector<int> vals = {10, 20, 30, 40, 50};

// Must define a custom order or use sorted keys
std::map<std::string, int> order = {{"A", 0}, {"B", 1}};

// Group and sum
std::map<std::string, int> sums;
for (size_t i = 0; i < cats.size(); ++i) {
    sums[cats[i]] += vals[i];
}

// Sort by custom order
std::vector<std::pair<std::string, int>> sorted(sums.begin(), sums.end());
std::sort(sorted.begin(), sorted.end(),
    [&](const auto& a, const auto& b) {
        return order[a.first] < order[b.first];
    });

// sorted = {{"A", 80}, {"B", 70}}
```

pandas manages category ordering, grouping, and output labeling natively.

### 6.3 Multi-Level GroupBy

**pandas:**
```python
mi = pd.MultiIndex.from_product([[1,2], ['red','blue']], names=['no','color'])
s = pd.Series([10, 20, 30, 40], index=mi)
s.groupby(level='no').sum()
```

**C++ equivalent (~30 lines):**
```cpp
#include <vector>
#include <tuple>
#include <map>

using Key = std::tuple<int, std::string>;
std::vector<std::pair<Key, int>> data = {
    {{1, "red"}, 10}, {{1, "blue"}, 20},
    {{2, "red"}, 30}, {{2, "blue"}, 40}
};

// Group by first element of the tuple
std::map<int, int> sums;
for (auto& [key, val] : data) {
    sums[std::get<0>(key)] += val;
}
// sums = {{1, 30}, {2, 70}}

// To group by second element ("color"), you'd need a separate loop
std::map<std::string, int> color_sums;
for (auto& [key, val] : data) {
    color_sums[std::get<1>(key)] += val;
}
```

In pandas, switching between grouping levels is a single parameter change (`level=0` vs `level=1`).  
In C++, you need separate extraction logic for each grouping dimension.

### Summary Table: pandas vs C++ (STL)

| Operation | pandas | C++ (STL) |
|-----------|--------|-----------|
| Date range generation | `pd.date_range(start, end, freq)` | Manual `mktime` loop or Boost.Chrono |
| Monthly resampling | `.resample('MS').mean()` | Manual month extraction + `std::accumulate` |
| Categorical groupby | `CategoricalIndex` + `.groupby().sum()` | `std::map` + manual ordering + accumulation |
| Multi-level index | `MultiIndex` + `level=` parameter | Nested maps or tuple-based keys |
| Timezone conversion | `.tz_localize()` / `.tz_convert()` | External library (Boost, Howard Hinnant's date) |
| Interval containment | `.contains()` / `.overlaps()` | Manual range checks with loops |
| Labeled output | Automatic with index/column names | Manual string formatting |

**Key advantage of pandas**: vectorized, labeled, indexed operations with zero boilerplate.  
**Key advantage of C++**: raw performance for compute-bound loops (but pandas uses NumPy/C under the hood).

---
## 7. Full Enterprise Scenario: Financial Portfolio Analysis

Combines **DatetimeIndex** (time series), **CategoricalIndex** (asset class), and **MultiIndex** (asset + date) in one cohesive example.

In [ ]:
np.random.seed(2025)

assets = {
    'AAPL':   {'class': 'Equity',    'base_price': 175},
    'GOOGL':  {'class': 'Equity',    'base_price': 140},
    'BOND-A': {'class': 'Fixed Income', 'base_price': 98},
    'BOND-B': {'class': 'Fixed Income', 'base_price': 102},
    'GOLD':   {'class': 'Commodity',  'base_price': 2000},
}

dates = pd.bdate_range('2025-01-01', '2025-06-30')
rows = []

for ticker, info in assets.items():
    n = len(dates)
    price = info['base_price'] + np.cumsum(np.random.randn(n) * 0.5)
    for d, p in zip(dates, price):
        rows.append({
            'Date': d,
            'Asset': ticker,
            'AssetClass': info['class'],
            'Price': round(p, 2),
            'Shares': np.random.randint(10, 500)
        })

portfolio = pd.DataFrame(rows)
portfolio['MarketValue'] = portfolio['Price'] * portfolio['Shares']

# Set up MultiIndex: (Date, Asset)
portfolio.set_index(['Date', 'Asset'], inplace=True)
portfolio.sort_index(inplace=True)

print('Portfolio data (first 10 rows):')
print(portfolio.head(10))

In [ ]:
# Total market value by asset class over time (resample monthly)
# First, unstack Asset to get per-asset columns, then resample
mv_by_class = portfolio.reset_index().groupby(
    [pd.Grouper(key='Date', freq='MS'), 'AssetClass']
)['MarketValue'].sum().unstack('AssetClass')

print('Monthly Market Value by Asset Class:')
print(mv_by_class.round(0).astype(int))

In [ ]:
# Cross-sectional view: portfolio on the last trading day
last_day = portfolio.index.get_level_values('Date').max()
end_of_period = portfolio.loc[last_day]

# Add CategoricalIndex for ordered display
asset_class_cat = pd.CategoricalIndex(
    end_of_period['AssetClass'],
    categories=['Equity', 'Fixed Income', 'Commodity'],
    ordered=True
)
end_of_period = end_of_period.copy()
end_of_period['AssetClass'] = asset_class_cat

print(f'Portfolio snapshot on {last_day.date()}:')
print(end_of_period.sort_values('AssetClass'))
print()

print('Total Market Value by Asset Class:')
print(end_of_period.groupby('AssetClass')['MarketValue'].sum().round(0).astype(int))

In [ ]:
# Time series: monthly returns per asset
prices_wide = portfolio['Price'].unstack('Asset')
monthly_returns = prices_wide.resample('MS').last().pct_change().dropna()

print('Monthly returns (%): first 5 months')
print((monthly_returns * 100).round(2))

In [ ]:
# Correlation matrix of monthly returns
print('Return correlation matrix:')
print(monthly_returns.corr().round(3))

---
## 8. Key Takeaways

1. **DatetimeIndex** is essential for time series -- `date_range`, `asfreq`, `resample`, timezone ops.
2. **CategoricalIndex** enforces category order and enables efficient group-by with ordered output.
3. **MultiIndex** handles hierarchical data natively -- `from_product`, `from_tuples`, `groupby(level=...)`.
4. **IntervalIndex** bins continuous data into labeled ranges (`cut`, `interval_range`).
5. **RangeIndex** is the efficient default integer index.
6. **vs C++**: pandas provides labeled, vectorized, indexed operations in minimal code. C++ (STL) requires manual container management, sorting, and accumulation loops for equivalent functionality.
7. **Enterprise use**: financial time series, HR analytics, supply-chain analysis all benefit from combining these index types.